In [1]:
from sympy import *
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb
%run CartanGeometry.ipynb

In [2]:
g=Symp_symb(7)
C=g.cochain_complex
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis
P=RegularCartanGeometry(g,'eta')
eta=IndexedBase('eta')
P.fund_invars=[eta[3,9,6],eta[6,9,9],eta[3,9,4]]
D=Distr_of_constant_symbol(g,-P.curvature)
P.curvature=ds_subs(P.curvature,{eta[3,9,6]:{tuple():0},eta[3,9,4]:{tuple():0}},D)[0]
D=Distr_of_constant_symbol(g,-P.curvature)

### Computations

In [3]:
tuples_by_wght={}
for i in range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        for k in range(j+1,len(g.basis)):
            for m in range(len(g.basis)):
                w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[m].wght
                if w not in tuples_by_wght: tuples_by_wght[w]=[]
                tuples_by_wght[w].append((i,j,k,m))

In [4]:
Bianchi_dict={}
not_added=[]
rel_Bianchi_dict={}
not_added_rel_Bianchi_dict={}

def compute_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        time0=time.time()
        print('Computing', t)
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else: 
            time1=time.time()
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
            print('    Bianchi computed in',hrs_min_sec(time.time()-time1))
        time2=time.time()
        to_solve,rel_keys=ds_subs(zero_elt.vec[m],Bianchi_dict,D)
        rel_Bianchi_keys=set()
        for a in rel_keys: rel_Bianchi_keys=rel_Bianchi_keys.union(rel_Bianchi_dict[a[0]][a[1]])
        rel_Bianchi_keys.add((i,j,k,m))

        print('    to_solve computed in',hrs_min_sec(time.time()-time2))
        if simplify(to_solve)!=0:
            time3=time.time()
            s=find_a_linear_term(to_solve,P.fund_invars)
            if s is None: s=find_a_linear_term(to_solve)
            if s is None:
                print('no linear term in',t)
                not_added.append(t)
                rel_Bianchi_dict[t]=rel_Bianchi_keys
            else:
                sol=solve(to_solve,s,dict=True)[0]
                print('    Solving complete in',hrs_min_sec(time.time()-time3))
                time4=time.time()
                for a in sol: 
                    ds_add_key(a,sol[a],Bianchi_dict,D,rel_Bianchi_keys,rel_Bianchi_dict,not_added)
                print('    Substitution complete in',hrs_min_sec(time.time()-time4))
        print('   ',t,'computed in',hrs_min_sec(time.time()-time0))
        # Notice that D.curv = -P.curvature, since I switched sign conventions

def check_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else:
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
        r=simplify(ds_subs(zero_elt.vec[m],Bianchi_dict,D)[0])
        if r!=0: print(r)

In [6]:
for w in range(1,10):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)[0]
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 1 ---------------------
Computing (3, 4, 5, 6)
    Bianchi computed in 3.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 5, 6) computed in 3.0 sec
Computing (3, 4, 6, 7)
    Bianchi computed in 3.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 6, 7) computed in 3.0 sec
Computing (3, 4, 7, 8)
    Bianchi computed in 6.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 7, 8) computed in 6.0 sec
Computing (3, 4, 8, 9)
    Bianchi computed in 11.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 8, 9) computed in 12.0 sec
Computing (3, 4, 9, 10)
    Bianchi computed in 33.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4,

In [ ]:
for w in range(1,10):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)[0]
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 1 ---------------------
Computing (3, 4, 5, 6)
    Bianchi computed in 4.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 5, 6) computed in 4.0 sec
Computing (3, 4, 6, 7)
    Bianchi computed in 3.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 6, 7) computed in 3.0 sec
Computing (3, 4, 7, 8)
    Bianchi computed in 7.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 7, 8) computed in 7.0 sec
Computing (3, 4, 8, 9)
    Bianchi computed in 12.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 8, 9) computed in 12.0 sec
Computing (3, 4, 9, 10)
    Bianchi computed in 32.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4,

In [8]:
# wght_8_Bianchi=copy.deepcopy(Bianchi_dict)
# Bianchi_dict=copy.deepcopy(wght_8_Bianchi)

In [9]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

False
[(4, 5, 10, 3)]


In [10]:
for w in range(10,13):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)[0]
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 10 ---------------------
Computing (3, 4, 10, 0)
    to_solve computed in 1.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 10, 0) computed in 2.0 sec
Computing (3, 5, 9, 0)
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 5, 9, 0) computed in 0.0 sec
Computing (3, 5, 10, 1)
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 5, 10, 1) computed in 1.0 sec
Computing (3, 5, 10, 2)
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 5, 10, 2) computed in 0.0 sec
Computing (3, 6, 8, 0)
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 6, 8, 0) computed in 0.0 sec
Computing (3, 6, 9, 1)
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (

In [11]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

False
[(4, 5, 10, 3), (4, 5, 10, 1), (4, 5, 10, 2), (4, 6, 10, 3), (4, 5, 10, 0), (4, 6, 10, 1), (4, 6, 10, 2), (4, 7, 10, 3), (5, 6, 10, 3), (4, 7, 10, 1), (4, 7, 10, 2), (4, 8, 10, 3), (5, 6, 10, 1), (5, 6, 10, 2), (5, 7, 10, 3)]


In [12]:
for w in range(13,17):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)[0]
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 13 ---------------------
Computing (3, 7, 10, 0)
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 7, 10, 0) computed in 0.0 sec
Computing (3, 8, 9, 0)
    to_solve computed in 0.0 sec
    (3, 8, 9, 0) computed in 0.0 sec
Computing (3, 8, 10, 1)
    to_solve computed in 1.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 8, 10, 1) computed in 1.0 sec
Computing (3, 8, 10, 2)
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 8, 10, 2) computed in 0.0 sec
Computing (3, 9, 10, 3)
    to_solve computed in 2.0 sec
    (3, 9, 10, 3) computed in 2.0 sec
Computing (3, 9, 10, 4)
    to_solve computed in 0.0 sec
    (3, 9, 10, 4) computed in 0.0 sec
Computing (4, 7, 10, 0)
    to_solve computed in 2.0 sec
    Solving complete in 1.0 sec
    Substitution complete in 0.0 sec
    (4, 7, 10, 0) computed in 4.0 sec
Computi

In [13]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

False
[(4, 5, 10, 3), (4, 5, 10, 1), (4, 5, 10, 2), (4, 6, 10, 3), (4, 5, 10, 0), (4, 6, 10, 1), (4, 6, 10, 2), (4, 7, 10, 3), (5, 6, 10, 3), (4, 7, 10, 1), (4, 7, 10, 2), (4, 8, 10, 3), (5, 6, 10, 1), (5, 6, 10, 2), (5, 7, 10, 3), (4, 8, 10, 1), (4, 8, 10, 2), (4, 9, 10, 3), (5, 6, 10, 0), (5, 7, 10, 1), (5, 7, 10, 2), (5, 8, 10, 3), (6, 7, 10, 3), (4, 8, 10, 0), (4, 9, 10, 1), (4, 9, 10, 2), (5, 7, 10, 0), (5, 8, 10, 1), (5, 8, 10, 2), (5, 9, 10, 3), (6, 7, 10, 1), (6, 7, 10, 2), (6, 8, 10, 3), (4, 9, 10, 0), (5, 8, 10, 0), (5, 9, 10, 1), (5, 9, 10, 2), (6, 7, 10, 0), (6, 8, 10, 1), (6, 8, 10, 2), (6, 9, 10, 3), (7, 8, 10, 3), (5, 9, 10, 0), (6, 8, 10, 0), (6, 9, 10, 1), (6, 9, 10, 2), (7, 8, 10, 1), (7, 8, 10, 2), (7, 9, 10, 3)]


In [14]:
for w in range(17,19):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature[0]
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 17 ---------------------
Computing (6, 9, 10, 0)
    to_solve computed in 3.0 sec
no linear term in (6, 9, 10, 0)
    (6, 9, 10, 0) computed in 3.0 sec
Computing (7, 8, 10, 0)
    to_solve computed in 2.0 sec
no linear term in (7, 8, 10, 0)
    (7, 8, 10, 0) computed in 2.0 sec
Computing (7, 9, 10, 1)
    to_solve computed in 2.0 sec
no linear term in (7, 9, 10, 1)
    (7, 9, 10, 1) computed in 3.0 sec
Computing (7, 9, 10, 2)
    to_solve computed in 2.0 sec
no linear term in (7, 9, 10, 2)
    (7, 9, 10, 2) computed in 2.0 sec
Computing (8, 9, 10, 3)
    to_solve computed in 3.0 sec
no linear term in (8, 9, 10, 3)
    (8, 9, 10, 3) computed in 3.0 sec
Computing (8, 9, 10, 4)
    to_solve computed in 0.0 sec
    (8, 9, 10, 4) computed in 0.0 sec
Bianchi wght 17 computed in 14.0 sec
Weight 17 complete in 14.0 sec
--------------------- Weight 18 ---------------------
Computing (7, 9, 10, 0)
    to_solve computed in 1.0 sec
no linear term in (7, 9, 10, 0)
    (

TypeError: bad operand type for unary -: 'tuple'

In [ ]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

### Syzygies

In [22]:
syzygies_by_wght={}
for k in P.fund_invars:
    if k in Bianchi_dict:
        for j in Bianchi_dict[k]:
            w=-wght_of_ind(k.base[k.indices+j],g)
            if w not in syzygies_by_wght: syzygies_by_wght[w]=[]
            new_syzygy=simplify(k.base[k.indices+j]-Bianchi_dict[k][j])
            if new_syzygy!=0:
                new_syzygy=new_syzygy.as_numer_denom()[0]
                syzygies_by_wght[w].append(new_syzygy)

In [ ]:
syzygies_by_wght.keys()

In [ ]:
for a in syzygies_by_wght[9]:
    display(a)

In [26]:
not_added_by_wght={}
for a in not_added:
    w0,w1,w2,w3=[g.basis[i].wght for i in a]
    w=-w0-w1-w2+w3
    if w not in not_added_by_wght: not_added_by_wght[w] = []
    not_added_by_wght[w].append(a)


In [27]:
def add_expr_to_subs_dict(expr,subs_dict):
    expr=expand(ds_subs(expr,Bianchi_dict,D).subs(subs_dict))
    if expr!=0:
        i=0
        new_key=expr.as_coeff_add()[1][i].as_coeff_Mul()[1]
        while type(new_key)==Pow and new_key.as_base_exp()[0]==eta[6,9,9]:
            i+=1
            new_key=expr.as_coeff_add()[1][i].as_coeff_Mul()[1]
        new_val=solve(expr,new_key)[0]
        back_subs({new_key:new_val},subs_dict)
        subs_dict[new_key]=new_val

def back_subs(new_dict,old_dict):
    for k in old_dict:
        old_dict[k]=old_dict[k].subs(new_dict)

In [ ]:
new_subs={}
for w in not_added_by_wght:
    for a in not_added_by_wght[w]:
        expr=simplify(ds_subs(P.Bianchi_cache[(a[0],a[1],a[2])].vec[a[3]],Bianchi_dict,D))
        add_expr_to_subs_dict(expr,new_subs)

In [ ]:
for a in not_added_by_wght[18]:
    display(simplify(ds_subs(P.Bianchi_cache[(a[0],a[1],a[2])].vec[a[3]],Bianchi_dict,D)))

### Branching

In [10]:
not_added_by_wght={}
for a in not_added:
    w0,w1,w2,w3=[g.basis[i].wght for i in a]
    w=-w0-w1-w2+w3
    if w not in not_added_by_wght: not_added_by_wght[w] = []
    not_added_by_wght[w].append(a)

In [ ]:
not_added_by_wght

In [ ]:
for a in not_added_by_wght[9]:
    display(simplify(ds_subs(P.Bianchi_cache[(a[0],a[1],a[2])].vec[a[3]],Bianchi_dict,D)))

### First branch

In [35]:
b1_Bianchi_dict=copy.deepcopy(Bianchi_dict)

In [37]:
ds_add_key(eta[6,9,9,4],0,b1_Bianchi_dict,D)

In [ ]:
ds_subs_needed(b1_Bianchi_dict)

In [ ]:
b1_Bianchi_dict[eta[6,9,9]]

### Second branch

In [12]:
b2_Bianchi_dict=copy.deepcopy(Bianchi_dict)

In [13]:
ds_add_key(eta[6,9,9,4,3],6*eta[6,9,9,3,4],b2_Bianchi_dict,D)

In [ ]:
ds_subs_needed(Bianchi_dict)

In [ ]:
b2_Bianchi_dict[eta[6,9,9]]

In [ ]:
# # Shelve
# with shelve.open('Abstract_Syzygies') as shelf:
#     shelf['Zero_Wilc_Bianchi_dict'+'{j}'.format(j=7)]=Bianchi_dict

# # Unshelve
# with shelve.open('Abstract_Syzygies') as shelf:
#     Bianchi_dict=shelf['Zero_Wilc_Bianchi_dict'+'{j}'.format(j=7)]

KeyError: b'Zero_Wilc_Bianchi_dict7'

### Ricci Identities

In [131]:
P1=RegularCartanGeometry(g,'eta')
P.curvature=ds_subs(P1.curvature,Bianchi_dict,D)[0]
simplify_cochain(P.curvature)
D.curv=-P.curvature

In [25]:
saved_Bianchi_dict=copy.deepcopy(Bianchi_dict) # Through wght 8+ at the moment
saved_curv=copy.deepcopy(P.curvature)

In [ ]:
duples_by_wght={}
for i in range(3,len(g.basis)):#range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        w=-g.basis[i].wght-g.basis[j].wght
        if w not in duples_by_wght:duples_by_wght[w]=[]
        duples_by_wght[w].append((i,j))

Ricci_dict={}
for w in range(2,7):
    for t in duples_by_wght[w]:
        i,j=t
        time0=time.time()
        temp=ds_subs(dds_subs(D.Ricci_Id(i,j),Ricci_dict,g,-D.curv),Bianchi_dict,D)
        for k in temp.d:
            temp.d[k]=simplify(temp.d[k])
        temp.clear_zeros()
        if temp.d!={}:
            print('Ricci_Id',(i,j),':',temp,'\n')
            Process_Ricci_Id(D,i,j,Ricci_dict,Bianchi_dict)
            print('\n',(i,j),'complete in time',hrs_min_sec(time.time()-time0),len(list(Ricci_dict.keys())))
            time1=time.time()
            for k in Ricci_dict:
                for j in Ricci_dict[k]:
                    Ricci_dict[k][j]=cancel(Ricci_dict[k][j])
            print('    simplification complete in time',hrs_min_sec(time.time()-time1),'\n')
            print('Ricci_dict:')
            for k in Ricci_dict:
                print('    ',k,'-->',Ricci_dict[k],'\n')
            print('------------------------------------------------------------\n')

In [58]:
def check_viability(t):
    i,j=t
    to_add=dds_subs(D.Ricci_Id(i,j),Ricci_dict,g,-D.curv)
    to_add=ds_subs(to_add,Bianchi_dict,D)
    viable=False
    for k in to_add.d:
        to_add.d[k]=simplify(to_add.d[k])
        if Indexed_obj_in_expr(to_add.d[k])==set(): viable=True
    return viable

remaining_pairs=set(duples_by_wght[7]+duples_by_wght[8]+duples_by_wght[9])

In [61]:
for t in list(remaining_pairs):
    if check_viability(t):
        i,j=t
        time0=time.time()
        temp=ds_subs(dds_subs(D.Ricci_Id(i,j),Ricci_dict,g,-D.curv),Bianchi_dict,D)
        for k in temp.d:
            temp.d[k]=simplify(temp.d[k])
        temp.clear_zeros()
        if temp.d!={}:
            print('Ricci_Id',(i,j),':',temp,'\n')
            Process_Ricci_Id(D,i,j,Ricci_dict,Bianchi_dict)
            print('\n',(i,j),'complete in time',hrs_min_sec(time.time()-time0),len(list(Ricci_dict.keys())))
            time1=time.time()
            for k in Ricci_dict:
                for j in Ricci_dict[k]:
                    Ricci_dict[k][j]=cancel(Ricci_dict[k][j])
            print('    simplification complete in time',hrs_min_sec(time.time()-time1),'\n')
            print('Ricci_dict:')
            for k in Ricci_dict:
                print('    ',k,'-->',Ricci_dict[k],'\n')
            print('------------------------------------------------------------\n')
        remaining_pairs.remove(t)

In [ ]:
remaining_pairs

In [ ]:
Ricci_dict.keys()

In [ ]:
Bianchi_dict[eta[6,9,9]]

In [ ]:
ds_subs(eta[6,9,9,3,3,4],Bianchi_dict,D)